In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))

def text_cleaning_pipeline(dataset, rule='lemmatize'):
    # Lowercase
    data = dataset.lower()
    # Remove URLs
    data = re.sub(r'https?://\S+|www\.\S+', '', data)
    # Remove emojis / non-ascii
    data = re.sub(r'[^\x00-\x7F]+', '', data)
    # Remove mentions and hashtags
    data = re.sub(r'@[A-Za-z0-9_]+', '', data)
    data = re.sub(r'#[A-Za-z0-9_]+', '', data)
    # Remove punctuation
    data = re.sub(r'[^a-z0-9 ]', '', data)
    # Tokenize
    tokens = data.split()
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    if rule == 'lemmatize':
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(t, pos='v') for t in tokens]
    elif rule == 'stem':
        ps = PorterStemmer()
        tokens = [ps.stem(t) for t in tokens]
    else:
        print('Pick between lemmatize or stem')
    return ' '.join(tokens)

text_cleaning_pipeline('Check out https://test.com @user #trump !!! Amazing news')

# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

Load dataset (sampled because full file is 2.4M rows)

In [ ]:
df = pd.read_csv('./data/trump_tweets_sentiment.csv', encoding='ISO-8859-1')
df = df.sample(20000, random_state=42).reset_index(drop=True)
df = df.rename(columns={'Sentiment': 'label'})
df = df.dropna(subset=['text', 'label'])
df.head()

In [ ]:
df['label'].value_counts()

Apply the text cleaning pipeline

In [ ]:
df['cleaned'] = df['text'].astype(str).apply(lambda t: text_cleaning_pipeline(t, rule='lemmatize'))
df[['text', 'cleaned']].head()

Train-test split

In [ ]:
X = df['cleaned']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

X_train_tfidf.shape

Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

Try on a new tweet

In [ ]:
sample = "I love what Trump is doing for the economy, amazing work!"
sample_clean = text_cleaning_pipeline(sample)
sample_vec = vectorizer.transform([sample_clean])
model.predict(sample_vec)[0]